# Distributed delay as a Volterra integral

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/31_distributed_delay_volterra.ipynb)

Logistic growth regulated by the population's whole past through a gamma kernel
K(s) = s e^(−s/τ)/τ² with mean delay 2τ, written as XPP's `int{K#x}`. The equilibrium x = 1
loses stability at r τ = 2.

*Based on an example from [XPP Studio](https://xpp-studio.github.io/xpp-studio/) (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.3.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("distributed_delay")
      .ode("x", "r*x*(1-int{t*exp(-t/tau)/tau^2#x})")
      .parameter(r=1, tau=2.5)
      .initial(x=0.5)
      .option(total=200, dt=0.05, vmaxpts=1000))
print(mb.render())

x'=r*x*(1-int{t*exp(-t/tau)/tau^2#x})
par r=1,tau=2.5
init x=0.5
@ total=200,dt=0.05,vmaxpts=1000
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="series", x="x", y="x", sliders={"tau": (0.5, 4), "r": (0.2, 2)}, title="Distributed delay")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Amplitude at long times against τ** (r = 1): onset near τ = 2.

In [5]:
import numpy as np
TAU = np.linspace(1.4, 3, 9)
amp = [np.ptp(ex.m.run(keep=False, tau=t, r=1, total=300, trans=200)["x"]) for t in TAU]
fig = go.Figure(go.Scatter(x=TAU, y=amp, mode="lines+markers")); fig.add_vline(x=2, line_dash="dot")
show_figure(fig, xaxis_title="tau (r = 1)", yaxis_title="amplitude", height=300)

## Things to try

- The Volterra method is slower than an ODE: `vmaxpts` sets how much history is kept.